# build_taxonomy_master_final

In [ ]:
# Builds a clean, complete taxonomy master from three sources (priority order):
#
#   1. cat_keys.csv                    - authoritative DB key list (source of truth)
#   2. taxonomy_master_updated_old.csv - manually curated enrichment (spend keys
#                                        + mixed keys with leans/manual_spend_override)
#   3. base_cat_keys_nonspend.csv      - confirmed non-spend metadata (spend_type,
#                                        Used in NS model, Removed flag)
#
# Logic:
#   - Start from cat_keys.csv (all valid keys, nothing added or removed)
#   - Exclude keys marked Removed = Yes in base_cat_keys_nonspend.csv
#   - Join enrichment from old taxonomy master where available
#   - Fill remaining gaps from base_cat_keys_nonspend.csv
#   - Small hardcoded fallback only for spend keys not in either source
#
# Output: taxonomy_master_final.csv

import pandas as pd
from pathlib import Path

CAT_KEYS_PATH = Path("assets/cat_keys.csv")
OLD_TAXONOMY  = Path("assets/taxonomy_master_updated_old.csv")
NS_KEYS_PATH  = Path("assets/base_cat_keys_nonspend.csv")
OUTPUT_PATH   = Path("assets/taxonomy_master_updated2.csv")


In [ ]:
# ============================================================
# 1. Load files
# ============================================================
cat_keys = pd.read_csv(CAT_KEYS_PATH)
old_tax  = pd.read_csv(OLD_TAXONOMY)
ns_keys  = pd.read_csv(NS_KEYS_PATH)

# Normalise cat_keys key column
key_col = "base_category_key" if "base_category_key" in cat_keys.columns else "key"
cat_keys = cat_keys[[key_col]].rename(columns={key_col: "base_category_key"})
cat_keys["base_category_key"] = cat_keys["base_category_key"].str.strip().str.upper()

# Normalise old taxonomy key column
if "category_key" in old_tax.columns:
    old_tax = old_tax.rename(columns={"category_key": "base_category_key"})
old_tax["base_category_key"] = old_tax["base_category_key"].str.strip().str.upper()

# Normalise ns_keys - key column is "Base Category"
ns_keys = ns_keys.rename(columns={"Base Category": "base_category_key"})
ns_keys["base_category_key"] = ns_keys["base_category_key"].str.strip().str.upper()
# Normalise Removed column - treat any "Yes" variant as removed
ns_keys["removed"] = ns_keys["Removed"].astype(str).str.strip().str.lower().str.startswith("yes")

print(f"cat_keys rows          : {len(cat_keys)}")
print(f"old taxonomy rows      : {len(old_tax)}")
print(f"base_cat_keys_nonspend : {len(ns_keys)} rows "
      f"({ns_keys['removed'].sum()} marked Removed)")

In [ ]:
# ============================================================
# 2. Flag removed keys - report but do not auto-exclude
#    (removal should be a conscious decision, not silent)
# ============================================================
removed_keys = set(ns_keys[ns_keys["removed"]]["base_category_key"])
removed_in_cat_keys = removed_keys & set(cat_keys["base_category_key"])

if removed_in_cat_keys:
    print(f"\nWARNING - keys marked Removed in base_cat_keys_nonspend "
          f"but still present in cat_keys.csv ({len(removed_in_cat_keys)}):")
    for k in sorted(removed_in_cat_keys):
        print(f"  {k}  <-- review: exclude from taxonomy_master_final?")
    print("  These are included below with a 'removed_flag' column set to True.")
    print("  Remove manually from the output CSV if confirmed deprecated.\n")

In [ ]:
# ============================================================
# 3. Build ns_keys enrichment lookup (non-removed rows only for clean enrichment)
#    Columns we extract: spend_type (always non_spend), Used in NS model
# ============================================================
ns_enrich = (
    ns_keys[~ns_keys["removed"]]
    [["base_category_key", "Used in NS model?"]]
    .copy()
)
ns_enrich["spend_type_ns"]    = "non_spend"
ns_enrich["used_in_ns_model"] = ns_enrich["Used in NS model?"].astype(str).str.strip()
ns_enrich = ns_enrich[["base_category_key", "spend_type_ns", "used_in_ns_model"]]


In [ ]:
# ============================================================
# 4. Left join enrichment layers onto cat_keys
# ============================================================
merged = cat_keys.merge(
    old_tax[["base_category_key", "spend_type", "leans",
             "manual_spend_override", "category_type", "category_group", "notes"]],
    on="base_category_key", how="left"
)

# Layer 2: fill spend_type from ns_keys where still missing
merged = merged.merge(ns_enrich, on="base_category_key", how="left")

mask_missing_spend = merged["spend_type"].isna()
merged.loc[mask_missing_spend, "spend_type"] = (
    merged.loc[mask_missing_spend, "spend_type_ns"]
)
merged.drop(columns=["spend_type_ns"], inplace=True)

# Add removed flag for transparency
merged["removed_flag"] = merged["base_category_key"].isin(removed_in_cat_keys)


In [ ]:
# ============================================================
# 5. Hardcoded fallback - only for spend keys genuinely not
#    covered by either source (should be a small residual set)
# ============================================================
SPEND_FALLBACKS = {
    # key                         : (category_group,      notes)
    "AUTOMOTIVE":                  ("Transport",           None),
    "GIFTS":                       ("Other",               None),
    "GYMS___FITNESS":              ("Health",              None),
    "OFFICE_MAINTENANCE":          ("Business",            None),
    "PERSONAL":                    ("Other",               None),
    "PETS_PET_CARE":               ("Other",               None),
    "RECREATION":                  ("Entertainment",       None),
    "RENT_PAID":                   ("Home",                None),
    "RESTAURANT":                  ("Food & Drink",        None),
    "TRAVEL_HOLIDAYS":             ("Entertainment",       None),
    "TRANSFER_BETWEEN_ACCOUNTS":   (None,                  "PO_REVIEW - confirm spend_type"),
    # fee keys - category_type only (spend_type already set via ns_keys)
    "FOREIGN_FEES":                (None,                  None),
    "LATE_FEES":                   (None,                  None),
    "OTHER_BILLS":                 (None,                  None),
    "OVERDRAWN_FEES":              (None,                  None),
    "OVERLIMIT_FEES":              (None,                  None),
    "REBATE_FEES":                 (None,                  None),
}

SPEND_ONLY_FALLBACKS = {
    k for k, v in SPEND_FALLBACKS.items() if k not in {
        "FOREIGN_FEES", "LATE_FEES", "OTHER_BILLS",
        "OVERDRAWN_FEES", "OVERLIMIT_FEES", "REBATE_FEES",
        "TRANSFER_BETWEEN_ACCOUNTS"
    }
}

for key, (grp, note) in SPEND_FALLBACKS.items():
    mask = merged["base_category_key"] == key
    if mask.sum() == 0:
        continue
    if merged.loc[mask, "spend_type"].isna().any():
        if key in SPEND_ONLY_FALLBACKS:
            merged.loc[mask, "spend_type"] = "spend"
    if grp and merged.loc[mask, "category_group"].isna().any():
        merged.loc[mask, "category_group"] = grp
    if note and merged.loc[mask, "notes"].isna().any():
        merged.loc[mask, "notes"] = note

# Fee keys - ensure category_type = FEES if not already set
fee_keys = {"FOREIGN_FEES", "LATE_FEES", "OTHER_BILLS",
            "OVERDRAWN_FEES", "OVERLIMIT_FEES", "REBATE_FEES"}
for key in fee_keys:
    mask = merged["base_category_key"] == key
    if mask.sum() > 0 and merged.loc[mask, "category_type"].isna().any():
        merged.loc[mask, "category_type"] = "FEES"

In [ ]:
# ============================================================
# 6. Validation report
# ============================================================
print("=" * 55)
print("VALIDATION")
print("=" * 55)

still_missing = merged[merged["spend_type"].isna()]["base_category_key"].tolist()
if still_missing:
    print(f"WARNING - spend_type still missing for {len(still_missing)} keys:")
    for k in still_missing:
        print(f"  {k}")
else:
    print(f"All {len(merged)} keys have spend_type - PASSED")

po_review = merged[merged["notes"].astype(str).str.contains("PO_REVIEW", na=False)]
if not po_review.empty:
    print(f"\nKeys flagged for PO review:")
    print(po_review[["base_category_key", "spend_type", "notes"]].to_string(index=False))

flagged_removed = merged[merged["removed_flag"]]
if not flagged_removed.empty:
    print(f"\nKeys flagged as Removed (confirm whether to exclude):")
    print(flagged_removed[["base_category_key"]].to_string(index=False))

print(f"\nFinal row count : {len(merged)}")
print(f"\nspend_type breakdown:")
print(merged["spend_type"].value_counts().to_string())

print(f"\nKeys still missing category_group (spend) or category_type (non-spend):")
spend_no_grp  = merged[(merged["spend_type"] == "spend") & merged["category_group"].isna()]
ns_no_type    = merged[(merged["spend_type"] == "non_spend") & merged["category_type"].isna()]
if spend_no_grp.empty and ns_no_type.empty:
    print("  None - PASSED")
else:
    if not spend_no_grp.empty:
        print(f"  Spend missing category_group: {spend_no_grp['base_category_key'].tolist()}")
    if not ns_no_type.empty:
        print(f"  Non-spend missing category_type: {ns_no_type['base_category_key'].tolist()}")

In [ ]:
# ============================================================
# 7. Save
# ============================================================
out_cols = ["base_category_key", "spend_type", "leans", "manual_spend_override",
            "category_type", "category_group", "used_in_ns_model",
            "removed_flag", "notes"]
merged[out_cols].to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")
print("Next step: review taxonomy_master_updated2.csv, then update Notebook 6 to load it.")